<a href="https://colab.research.google.com/github/Vania-cortina/Reconocimiento-de-Emociones/blob/main/Reconocimiento_de_emociones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Activacion de  GPU

In [ ]:
!nvidia-smi

Tue Nov 25 15:30:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Instalar dependencias

In [ ]:
!pip install tensorflow keras opencv-python-headless numpy pandas scikit-learn matplotlib seaborn kaggle streamlit

Conectar con Kaggle para descargar datasets

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = '/content'
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat '/content/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d shawon10/ckplus
!kaggle datasets download -d msambare/fer2013
!kaggle datasets download -d ashwingupta3012/rafdb

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'
Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File 

In [ ]:
!unzip ckplus.zip -d /content/data/raw/ckplus
!unzip fer2013.zip -d /content/data/raw/fer2013
!unzip rafdb.zip -d /content/data/raw/rafdb

unzip:  cannot find or open ckplus.zip, ckplus.zip.zip or ckplus.zip.ZIP.
unzip:  cannot find or open fer2013.zip, fer2013.zip.zip or fer2013.zip.ZIP.
unzip:  cannot find or open rafdb.zip, rafdb.zip.zip or rafdb.zip.ZIP.


In [ ]:
# 1) Instalar dependencias
!pip install tensorflow keras opencv-python-headless numpy pandas scikit-learn matplotlib seaborn kaggle streamlit

# Ver versiones (opcional)
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [ ]:
# 2.1) Configurar kaggle.json (asegúrate de haber subido kaggle.json al /content)
import os
if os.path.exists('/content/kaggle.json'):
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp /content/kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
    print("kaggle.json instalado correctamente.")
else:
    print("No se encontró /content/kaggle.json. Sube kaggle.json al panel de Archivos y vuelve a ejecutar esta celda.")

kaggle.json instalado correctamente.


In [ ]:
# 2.2) Descargar datasets desde Kaggle
# Estos comandos descargan los datasets a /content/data/raw/
import os
os.makedirs('/content/data/raw', exist_ok=True)

datasets = [
    "shawon10/ckplus",
    "msambare/fer2013",
    "ashwingupta3012/rafdb"
]

for d in datasets:
    print("Descargando:", d)
    !kaggle datasets download -d {d} -p /content/data/raw --unzip
print("Descargas completas.")

Descargando: shawon10/ckplus
Dataset URL: https://www.kaggle.com/datasets/shawon10/ckplus
License(s): CC0-1.0
  0% 0.00/3.63M [00:00<?, ?B/s]
100% 3.63M/3.63M [00:00<00:00, 943MB/s]
Descargando: msambare/fer2013
Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
  0% 0.00/60.3M [00:00<?, ?B/s]
100% 60.3M/60.3M [00:00<00:00, 1.80GB/s]
Descargando: ashwingupta3012/rafdb
403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/ashwingupta3012/rafdb
Descargas completas.


Preprocesamiento

In [ ]:
!kaggle datasets download -d deadskull7/fer2013
!unzip fer2013.zip -d /content/data/raw/fer2013


Dataset URL: https://www.kaggle.com/datasets/deadskull7/fer2013
License(s): CC0-1.0
  0% 0.00/96.6M [00:00<?, ?B/s]
100% 96.6M/96.6M [00:00<00:00, 1.79GB/s]
Archive:  fer2013.zip
  inflating: /content/data/raw/fer2013/fer2013.csv  


In [ ]:
# 3) Funciones de preprocesamiento y carga básica
import cv2
import numpy as np
import os
from glob import glob
from tqdm import tqdm
import pandas as pd

TARGET_SIZE = (48,48)

def preprocess_img(img_path, target_size=TARGET_SIZE):
    img = cv2.imdecode(np.fromfile(img_path, dtype=np.uint8), cv2.IMREAD_COLOR) if isinstance(img_path, str) else img_path
    # if imdecode used, may return None if path invalid; handle externally
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, target_size)
    normalized = resized.astype('float32') / 255.0
    return normalized

def load_fer2013(csv_path='/content/data/raw/fer2013/fer2013.csv'):
    # FER2013 viene en CSV con columnas: emotion, pixels, Usage
    if not os.path.exists(csv_path):
        print('FER2013 CSV no encontrado en', csv_path)
        return np.array([]), np.array([])
    df = pd.read_csv(csv_path)
    X = []
    y = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        pixels = np.fromstring(row['pixels'], sep=' ', dtype='float32')
        if pixels.size != 48*48:
            continue
        img = pixels.reshape(48,48)
        img = img / 255.0
        X.append(img)
        y.append(int(row['emotion']))
    X = np.array(X).reshape(-1,48,48,1)
    y = np.array(y)
    print('FER2013 cargado:', X.shape, y.shape)
    return X, y

def load_ckplus(root_dir='/content/data/raw/ckplus'):
    # CK+ suele venir en carpetas con imágenes; intentamos buscar .png/.jpg
    X, y = [], []
    # Asumir subcarpetas o archivos directos; buscar en profundidad
    img_paths = []
    for ext in ('**/*.png','**/*.jpg','**/*.jpeg'):
        img_paths += glob(os.path.join(root_dir, ext), recursive=True)
    img_paths = sorted(set(img_paths))
    # CK+ labeling puede requerir parsing de archivos de etiqueta; sin embargo,
    # muchos zips en Kaggle contienen carpetas por emoción o archivos con sufijos.
    # Aquí cargamos imágenes y dejaremos etiquetas como UNKNOWN (-1) para mezclar luego.
    for p in tqdm(img_paths[:5000]):  # límite por seguridad si hay muchas
        img = preprocess_img(p)
        if img is None:
            continue
        X.append(img)
        y.append(-1)  # etiqueta desconocida; se recomienda etiquetar correctamente si se desea
    if len(X)==0:
        return np.array([]), np.array([])
    X = np.array(X).reshape(-1,48,48,1)
    y = np.array(y)
    print('CK+ (sin etiquetas fiables) cargado:', X.shape)
    return X, y

def load_rafdb(root_dir='/content/data/raw/RAF-DB'):
    # RAF-DB en Kaggle puede venir en structure diferente; intentaremos buscar imágenes
    X, y = [], []
    img_paths = []
    for ext in ('**/*.png','**/*.jpg','**/*.jpeg'):
        img_paths += glob(os.path.join(root_dir, ext), recursive=True)
    img_paths = sorted(set(img_paths))
    for p in tqdm(img_paths[:10000]):
        img = preprocess_img(p)
        if img is None:
            continue
        X.append(img)
        y.append(-1)
    if len(X)==0:
        return np.array([]), np.array([])
    X = np.array(X).reshape(-1,48,48,1)
    y = np.array(y)
    print('RAF-DB (sin etiquetas procesadas) cargado:', X.shape)
    return X, y

# Nota: CK+ y RAF-DB podrían necesitar parsing de etiquetas específicas dependiendo del zip descargado.
# FER2013 es directo si el CSV está presente.


Combinar DataSets

In [ ]:
# Cargar FER2013 y preparar train/test/val usando solo FER2013 por ahora
X, y = load_fer2013('/content/fer2013.csv')

from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

FER2013 CSV no encontrado en /content/fer2013.csv


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

Definir la CNN

In [ ]:
# 4) Definir el modelo
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

num_classes = 7  # FER2013 tiene 7 clases

def build_emotion_model(input_shape=(48,48,1), num_classes=7):
    model = Sequential()
    model.add(Conv2D(32,(3,3), activation='relu', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))
    model.add(Conv2D(64,(3,3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))
    model.add(Conv2D(128,(3,3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))
    model.add(Flatten())
    model.add(Dropout(0.5))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_emotion_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 46, 46, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 21, 21, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 356,743 (1.36 MB)

 Trainable params: 356,295 (1.36 MB)

 Non-trainable params: 448 (1.75 KB)

Entrenamiento

In [ ]:
# 4) Compilar modelo (asegúrate de tener esto antes de entrenar)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # 🔹 Cambiado de sparse_categorical_crossentropy
    metrics=['accuracy']
)


In [ ]:
# 5) Entrenar el modelo
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

batch_size = 64
epochs = 25

# DATA AUGMENTATION
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

train_gen = datagen.flow(X_train, y_train, batch_size=batch_size)
val_gen = ImageDataGenerator().flow(X_val, y_val, batch_size=batch_size)

# COMPILAR MODELO (CORREGIDO)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # CORREGIDO
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint('/content/facial_emotion_model.h5',
                             monitor='val_accuracy',
                             save_best_only=True,
                             verbose=1)

early = EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)

history = model.fit(
    train_gen,
    epochs=epochs,
    validation_data=val_gen,
    callbacks=[checkpoint, early]
)


Epoch 1/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.2639 - loss: 2.0342
Epoch 1: val_accuracy improved from -inf to 0.32126, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 28s 47ms/step - accuracy: 0.2640 - loss: 2.0337 - val_accuracy: 0.3213 - val_loss: 1.7274
Epoch 2/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3744 - loss: 1.5939
Epoch 2: val_accuracy improved from 0.32126 to 0.44692, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.3744 - loss: 1.5938 - val_accuracy: 0.4469 - val_loss: 1.4376
Epoch 3/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4227 - loss: 1.4914
Epoch 3: val_accuracy did not improve from 0.44692
449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.4228 - loss: 1.4913 - val_accuracy: 0.4352 - val_loss: 1.4675
Epoch 4/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.4515 - loss: 1.4191
Epoch 4: val_accuracy improved from 0.44692 to 0.46057, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 37ms/step - accuracy: 0.4516 - loss: 1.4191 - val_accuracy: 0.4606 - val_loss: 1.4067
Epoch 5/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4786 - loss: 1.3657
Epoch 5: val_accuracy improved from 0.46057 to 0.46531, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.4785 - loss: 1.3656 - val_accuracy: 0.4653 - val_loss: 1.4027
Epoch 6/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.4893 - loss: 1.3355
Epoch 6: val_accuracy improved from 0.46531 to 0.50961, saving model to /content/facial_emotion_model.h5


...

449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.5213 - loss: 1.2576 - val_accuracy: 0.5252 - val_loss: 1.2517
Epoch 11/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5300 - loss: 1.2316
Epoch 11: val_accuracy improved from 0.52522 to 0.56199, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.5300 - loss: 1.2316 - val_accuracy: 0.5620 - val_loss: 1.1599
Epoch 12/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5363 - loss: 1.2212
Epoch 12: val_accuracy did not improve from 0.56199
449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.5363 - loss: 1.2213 - val_accuracy: 0.5492 - val_loss: 1.1911
Epoch 13/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5374 - loss: 1.2112
Epoch 13: val_accuracy did not improve from 0.56199
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.5374 - loss: 1.2112 - val_accuracy: 0.5447 - val_loss: 1.2086
Epoch 14/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5433 - loss: 1.2127
Epoch 14: val_accuracy did not improve from 0.56199
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.5433 - loss: 1.2127 - val_accuracy: 0.5372 - val_loss: 1.2092
Epoch 15/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5488 - loss: 1.1990
Epoch 15: val_ac

449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.5557 - loss: 1.1824 - val_accuracy: 0.5879 - val_loss: 1.0966
Epoch 17/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5538 - loss: 1.1764
Epoch 17: val_accuracy did not improve from 0.58791
449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.5538 - loss: 1.1764 - val_accuracy: 0.5868 - val_loss: 1.1071
Epoch 18/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5549 - loss: 1.1761
Epoch 18: val_accuracy did not improve from 0.58791
449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.5549 - loss: 1.1761 - val_accuracy: 0.5740 - val_loss: 1.1028
Epoch 19/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5602 - loss: 1.1606
Epoch 19: val_accuracy did not improve from 0.58791
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.5602 - loss: 1.1607 - val_accuracy: 0.5857 - val_loss: 1.1238
Epoch 20/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5592 - loss: 1.1595
Epoch 20: val_ac

449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.5618 - loss: 1.1506 - val_accuracy: 0.5949 - val_loss: 1.0683
Epoch 22/25
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5678 - loss: 1.1502
Epoch 22: val_accuracy improved from 0.59487 to 0.59627, saving model to /content/facial_emotion_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.5678 - loss: 1.1502 - val_accuracy: 0.5963 - val_loss: 1.0844
Epoch 23/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5737 - loss: 1.1280
Epoch 23: val_accuracy did not improve from 0.59627
449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.5737 - loss: 1.1281 - val_accuracy: 0.5717 - val_loss: 1.1321
Epoch 24/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5711 - loss: 1.1415
Epoch 24: val_accuracy did not improve from 0.59627
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.5711 - loss: 1.1415 - val_accuracy: 0.5818 - val_loss: 1.1021
Epoch 25/25
448/449 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5756 - loss: 1.1266
Epoch 25: val_accuracy did not improve from 0.59627
449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 33ms/step - accuracy: 0.5756 - loss: 1.1266 - val_accuracy: 0.5832 - val_loss: 1.0840


Mini App con Streamlit

In [ ]:
# 7) Crear app Streamlit
app_code = r'''
import streamlit as st
from tensorflow.keras.models import load_model
import numpy as np
from PIL import Image
import cv2

model = load_model('/content/facial_emotion_model.h5')
emotions = ['Angry','Disgust','Fear','Happy','Sad','Surprise','Neutral']  # ajustar orden según etiqueta

st.title("Facial Emotion Recognition — Demo")

uploaded_file = st.file_uploader("Sube una imagen", type=['png','jpg','jpeg'])
if uploaded_file:
    img = Image.open(uploaded_file).convert('RGB')
    st.image(img, caption='Imagen subida', use_column_width=True)
    st.write("")
    st.write("Prediciendo...")
    img_arr = np.array(img)
    gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (48,48)).astype('float32')/255.0
    inp = resized.reshape(1,48,48,1)
    preds = model.predict(inp)[0]
    top_idx = preds.argmax()
    st.subheader(f"Emoción detectada: {emotions[top_idx]} (confianza: {preds[top_idx]:.2f})")
    st.bar_chart({'probabilidad': preds})
'''
with open('/content/app.py','w') as f:
    f.write(app_code)
print('app.py creado en /content/app.py')

app.py creado en /content/app.py


Ejecutar Streamlit

In [ ]:
!pip install pyngrok

In [ ]:
# 7.1) Ejecutar Streamlit + ngrok
# Asegúrate de haber creado /content/app.py y tener /content/facial_emotion_model.h5
!ngrok config add-authtoken 35MJSCKYxRXSAiR2xYyJoa7eHrr_69PCWPSyN71ThsjrEaERg
from pyngrok import ngrok
import subprocess, time, os, signal

# Cerrar conexiones previas
ngrok.kill()

# Ejecutar streamlit en background
cmd = "streamlit run /content/app.py --server.port 8501"
proc = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Abrir tunel ngrok
public_url = ngrok.connect(8501).public_url
print("Streamlit corriendo en:", public_url)
print("Nota: mantén esta celda en ejecución para que la app siga disponible.")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Streamlit corriendo en: https://noneradicative-prefuneral-tobi.ngrok-free.dev
Nota: mantén esta celda en ejecución para que la app siga disponible.
